# 07.5 — RAG from Scratch

**RAG (Retrieval-Augmented Generation):** Retrieve relevant passages from a knowledge base, then inject them into a generator's context. Combines the factual grounding of retrieval with the fluency of generation.

**Why RAG?**
- LLMs hallucinate facts that aren't in their training data
- Knowledge in weights is frozen at training time
- RAG = dynamic, updatable knowledge without retraining

**Pipeline (no LangChain, no abstractions):**
1. Chunk documents
2. Embed chunks → vector store
3. At query time: embed query → retrieve top-k chunks
4. Inject chunks into LLM prompt → generate answer

---

In [ ]:
import re
import math
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass, field

# ---- Step 1: Document Chunking ----
# Chunking is underrated — bad chunking = bad retrieval regardless of embeddings.

@dataclass
class Chunk:
    text: str
    doc_id: str
    chunk_id: int
    start_char: int
    metadata: dict = field(default_factory=dict)


def fixed_size_chunk(text: str, doc_id: str, 
                     chunk_size=200, overlap=50) -> List[Chunk]:
    """
    Split text into fixed-size token windows with overlap.
    Overlap ensures context at chunk boundaries isn't lost.
    chunk_size and overlap in characters (approximate tokens).
    """
    chunks = []
    start = 0
    chunk_id = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        # Extend to nearest sentence boundary
        if end < len(text):
            # Find last sentence end in the window
            boundary = text.rfind('.', start, end)
            if boundary > start:
                end = boundary + 1
        chunks.append(Chunk(
            text=text[start:end].strip(),
            doc_id=doc_id,
            chunk_id=chunk_id,
            start_char=start,
        ))
        start = end - overlap
        chunk_id += 1
    return [c for c in chunks if len(c.text) > 20]  # filter tiny chunks


def semantic_chunk(text: str, doc_id: str, 
                   min_size=100, max_size=500) -> List[Chunk]:
    """
    Split on paragraph/heading boundaries first, then further split
    if a paragraph exceeds max_size.
    Respects natural document structure.
    """
    # Split on double newlines (paragraph breaks)
    paragraphs = re.split(r'\n{2,}', text.strip())
    chunks = []
    chunk_id = 0
    char_pos = 0
    for para in paragraphs:
        para = para.strip()
        if not para:
            char_pos += len(para) + 2
            continue
        if len(para) <= max_size:
            if len(para) >= min_size:
                chunks.append(Chunk(para, doc_id, chunk_id, char_pos))
                chunk_id += 1
        else:
            # Sub-chunk large paragraphs
            sub_chunks = fixed_size_chunk(para, doc_id, max_size, 50)
            for sc in sub_chunks:
                sc.chunk_id = chunk_id
                sc.start_char += char_pos
                chunks.append(sc)
                chunk_id += 1
        char_pos += len(para) + 2
    return chunks


# Test chunking
sample_doc = """
The transformer architecture was introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017.
It replaced recurrent and convolutional networks for sequence modeling tasks.

The key innovation is the self-attention mechanism, which allows the model to attend to all positions in the
input simultaneously. This is fundamentally different from RNNs which process tokens sequentially.

BERT, introduced by Devlin et al. in 2018, applies bidirectional transformers for pre-training on unlabeled text.
It achieves state-of-the-art results on many NLP benchmarks by fine-tuning on task-specific data.

GPT uses a decoder-only transformer with causal language modeling. Unlike BERT, it generates text autoregressively.
GPT-3 has 175 billion parameters and can perform few-shot learning from natural language prompts.
""".strip()

chunks = semantic_chunk(sample_doc, doc_id="nlp_intro")
print(f"Document split into {len(chunks)} chunks:")
for c in chunks:
    print(f"  [{c.chunk_id}] {c.text[:80]}...")

In [ ]:
# ---- Step 2: Embedding + Vector Store (numpy-based) ----
# In production: use a real embedding model (OpenAI text-embedding-3,
# sentence-transformers/all-MiniLM) + FAISS.
# Here: TF-IDF embeddings so we don't need a GPU.

from collections import Counter

class TFIDFEmbedder:
    """Character-level TF-IDF as a stand-in for a neural embedder."""
    def __init__(self, max_features=500):
        self.max_features = max_features
        self.vocab = None
        self.idf = None
    
    def _tokenize(self, text):
        return re.findall(r'\b[a-z]+\b', text.lower())
    
    def fit(self, texts):
        tokenized = [self._tokenize(t) for t in texts]
        # Document frequency
        df = Counter()
        for tokens in tokenized:
            df.update(set(tokens))
        # Keep top features by df
        N = len(texts)
        top_words = [w for w, _ in df.most_common(self.max_features)]
        self.vocab = {w: i for i, w in enumerate(top_words)}
        self.idf = np.array([
            math.log((N + 1) / (df[w] + 1)) + 1 for w in top_words
        ])
        return self
    
    def embed(self, text):
        tokens = self._tokenize(text)
        tf = Counter(tokens)
        vec = np.zeros(len(self.vocab))
        for w, count in tf.items():
            if w in self.vocab:
                vec[self.vocab[w]] = (1 + math.log(count)) * self.idf[self.vocab[w]]
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec


class VectorStore:
    """Simple in-memory vector store."""
    def __init__(self, embedder):
        self.embedder = embedder
        self.chunks: List[Chunk] = []
        self.embeddings: np.ndarray = None
    
    def add_chunks(self, chunks: List[Chunk]):
        texts = [c.text for c in chunks]
        self.embedder.fit(texts)
        embs = np.array([self.embedder.embed(t) for t in texts])
        self.chunks = chunks
        self.embeddings = embs
        print(f"Indexed {len(chunks)} chunks, embedding shape: {embs.shape}")
    
    def search(self, query: str, k=3) -> List[Tuple[Chunk, float]]:
        q_emb = self.embedder.embed(query)
        scores = self.embeddings @ q_emb
        top_k = np.argsort(-scores)[:k]
        return [(self.chunks[i], float(scores[i])) for i in top_k]


# Build the index
embedder = TFIDFEmbedder(max_features=200)
store = VectorStore(embedder)
store.add_chunks(chunks)

# Test retrieval
results = store.search("how does self attention work", k=2)
print("\nQuery: 'how does self attention work'")
for chunk, score in results:
    print(f"  score={score:.4f}: {chunk.text[:100]}...")

In [ ]:
# ---- Step 3: Generation with context injection ----
# We'll show the prompt template that would be sent to an LLM.
# Without an LLM API key here, we show the full pipeline structure.

def build_rag_prompt(query: str, retrieved_chunks: List[Tuple[Chunk, float]], 
                     max_context_chars=1500) -> str:
    """
    Construct the prompt for the LLM.
    
    RAG prompt structure:
    1. System instruction (be grounded, cite sources)
    2. Retrieved context passages
    3. User question
    4. Answer instruction
    """
    context_parts = []
    total_chars = 0
    for i, (chunk, score) in enumerate(retrieved_chunks):
        if total_chars + len(chunk.text) > max_context_chars:
            break
        context_parts.append(f"[Source {i+1}, doc={chunk.doc_id}]\n{chunk.text}")
        total_chars += len(chunk.text)
    
    context_str = "\n\n".join(context_parts)
    
    prompt = f"""You are a helpful assistant that answers questions based on the provided context.
Only use information from the context. If the answer is not in the context, say so.

CONTEXT:
{context_str}

QUESTION: {query}

ANSWER:"""
    return prompt


# Full RAG pipeline demo
def rag_query(query: str, store: VectorStore, k=3, verbose=True):
    # 1. Retrieve
    retrieved = store.search(query, k=k)
    # 2. Build prompt
    prompt = build_rag_prompt(query, retrieved)
    if verbose:
        print(f"Query: {query!r}")
        print(f"\nRetrieved {len(retrieved)} chunks:")
        for chunk, score in retrieved:
            print(f"  [score={score:.3f}] {chunk.text[:80]}...")
        print(f"\nPrompt ({len(prompt)} chars):")
        print(prompt[:500] + "..." if len(prompt) > 500 else prompt)
    return prompt, retrieved


prompt, retrieved = rag_query("What is BERT and how does it work?", store)

In [ ]:
# ---- Advanced RAG patterns ----

print("=== RAG Failure Modes & Solutions ===")
print()
print("1. RETRIEVAL FAILURE (right answer in corpus, not retrieved)")
print("   Symptoms: LLM says 'I don't have info' but corpus has it")
print("   Fixes:")
print("     - HyDE: generate a hypothetical answer, embed it, retrieve on that")
print("     - Query expansion: rewrite query as multiple sub-queries")
print("     - Hybrid BM25+dense + RRF")
print()
print("2. CONTEXT OVERFLOW (too many chunks, LLM ignores some)")
print("   Symptoms: LLM uses first/last chunks, ignores middle (lost in middle problem)")
print("   Fixes:")
print("     - Re-rank: use a cross-encoder to re-order retrieved chunks")
print("     - Compression: extract only the relevant sentences from each chunk")
print("     - Reduce k, increase chunk quality")
print()
print("3. CONTEXT POLLUTION (irrelevant chunks confuse the LLM)")
print("   Symptoms: LLM contradicts itself or mixes info from unrelated chunks")
print("   Fixes:")
print("     - Relevance threshold: only pass chunks above a minimum score")
print("     - Reranker to filter")
print()
print("4. STALE DATA")
print("   Fixes: incremental indexing, metadata filtering by date")

print()
print("=== HyDE: Hypothetical Document Embeddings ===")
print()
print("Query: 'How does attention work in transformers?'")
print()
print("HyDE step 1: LLM generates hypothetical answer:")
print("  'Attention in transformers works by computing query, key, and value")
print("   vectors from the input. The similarity between query and key determines")
print("   how much each value contributes to the output.'")
print()
print("HyDE step 2: Embed the hypothetical answer (not the original query)")
print("HyDE step 3: Retrieve passages similar to the hypothetical answer")
print()
print("Why it works: hypothetical answers use domain vocabulary → better embeddings")

In [ ]:
# ---- Chunking strategies comparison ----

print("Chunking strategy comparison:")
print()

strategies = [
    ("Fixed-size (no overlap)",
     "Simple, predictable size",
     "Cuts mid-sentence, loses context at boundaries"),
    ("Fixed-size (with overlap)",
     "Mitigates boundary problem",
     "Duplicate content in index, higher storage"),
    ("Sentence-based",
     "Preserves semantic units",
     "Highly variable chunk size, short sentences = weak embeddings"),
    ("Paragraph-based",
     "Best alignment with natural structure",
     "Requires clean paragraph breaks (bad for PDFs)"),
    ("Recursive (LangChain-style)",
     "Tries paragraph→sentence→word boundaries in order",
     "Complex; depends on separator hierarchy"),
    ("Semantic (embedding similarity)",
     "Groups sentences with similar meaning",
     "Slow; requires embedding every sentence"),
]

for name, pros, cons in strategies:
    print(f"  {name}")
    print(f"    + {pros}")
    print(f"    - {cons}")
    print()

print("Rule of thumb: start with paragraph chunking (200-500 tokens),")
print("10-20% overlap. Profile retrieval quality before optimizing.")

## Summary

**RAG pipeline:** Chunk → Embed → Index → (Query → Retrieve → Prompt → Generate)

**The hardest parts (in practice):**
1. Chunking: document parsing quality determines retrieval quality
2. Embedding model choice: domain-specific models beat general ones
3. Evaluation: you need labeled question-answer pairs to measure retrieval recall and generation correctness separately

**The 80/20:** Good chunking + BM25 hybrid often beats complex neural pipelines. Add cross-encoder re-ranking if you have latency budget.

---
**Next:** 07.6 — LoRA Fine-tuning